In [22]:
import math, pandas as pd
from math import gcd
from collections import Counter
from scipy.stats import chi2

def primeFactors(n):
    factors = set()
    i=2
    while i * i <= n:
        if n % i == 0:
            while n % i == 0:
                n //= i
        i += 1
    if n > 1:
        factors.add(n)
    return factors

#Test Hull-Dobbell (Vérifie si une séquence pseudo aléatoire est de période maximum)
def isFullPeriod(a, c, m):
    rule1 = gcd(c,m) == 1

    primeM = primeFactors(m)
    rule2 = all((a-1)%p == 0 for p in primeM)

    rule3 = (m % 4 !=0) or ((a-1) % 4 == 0)
    return rule1 and rule2 and rule3

#Formule congruentiel linéaire mixte (Générer la suite pseudo aléatoire)
def xnCompute(a, c , m, x0):
    xn = []
    xn.append(x0)
    for i in range(m-1):
        xn.append(((a * xn[i]) + c) % m)
    return xn

#Test des fréquence
def unCompute(a,c,m,x0):
    xn = xnCompute(a,c,m,x0)
    un = []
    for i in range(len(xn)):
        un.append(xn[i] / m)
    return un

#Fréquence cumulée
def ynCompute(a,c,m,x0):
    un = unCompute(a,c,m,x0)
    yn =[]
    for i in range(len(un)):
        yn.append(int(un[i]*10))
    return yn

#test de saut (Savoir l'espace entre chaque nombre demandé dans la suite)
def jumpTest(a,c,m,x0, studiedNb):
    yn = ynCompute(a,c,m,x0)
    jump = []
    try:
        iPosition = yn.index(studiedNb)
    except ValueError:
        return jump 
    print(iPosition)

    for i in range(iPosition + 1, len(yn)):
        if(yn[i] == studiedNb):
            jump.append(i - iPosition - 1)
            iPosition = i
    return jump

#Test de course (Comparé les nombre 2 a 2 si le premier est supérieur au 2ème = 1 et si 1er supérieur = 2)
def courseTest(a, c, m, x0, size):
    xn = xnCompute(a, c, m, x0)
    course = []
    for i in range(0, size*2, 2):
        if xn[i] > xn[i + 1]:
            course.append(2)
        else:
            course.append(1)
    return course

#Permet de séparer le jeu de données and x groupe d'une taille y
def separating(a, c, m, x0, size, nbGroup):
    yn = ynCompute(a, c, m, x0)
    separated = []
    
    for i in range(0, size*nbGroup, size):
        temp = []
        for i in range(size):
            temp.append(yn[i])
        separated.append(temp)
    return separated

#compte le nombre de chaque combinaison possible dans un test de poker
def pokerCount(a,c,m,x0):
    suite = ynCompute(a,c,m,x0)
    poker = []
    #Dans l'ordre(Poker, Carré, Full, Brelan, Deux pair, une pair, rien)
    pokerCounter = [0,0,0,0,0,0,0]
    for i in range(0,len(suite), 5):
        if i + 5 <= len(suite):
            group= []
            for j in range(5):
                group.append(suite[i+j])
            poker.append(pokerCheck(group))
    for i in range(len(poker)):
        if poker[i] == "Poker":
            pokerCounter[0]+=1
        elif poker[i] == "Carré":
            pokerCounter[1]+=1
        elif poker[i] == "Full":
            pokerCounter[2]+=1
        elif poker[i] == "Brelan":
            pokerCounter[3]+=1
        elif poker[i] == "Deux Pair":
            pokerCounter[4]+=1
        elif poker[i] == "Une Pair":
            pokerCounter[5]+=1
        else:
            pokerCounter[6]+=1
    return pokerCounter

#Poker (dans groupe de 5 vérifie si il y a soit une pair, soit deux pair, soit un brelan(3 les memes) soit un carré(4 les memes) soit un full (une pair + un brelan) soit un poker(5 les memes) soit rien)
def pokerCheck(group):
    count = Counter(group)
    value = count.values()
    if 5 in value:
        return "Poker"
    elif 4 in value:
        return "Carré"
    elif 3 in value and 2 in value:
        return "Full"
    elif 3 in value:
        return "Brelan"
    elif list(value).count(2) ==2:
        return "Deux Pair"
    elif 2 in value:
        return "Une Pair"
    else:
        return "Rien"

# Compte combien de fois chaque nombre apparaît dans le test de course
def counting(course):
    counts = {}
    for value in course:
        if value in counts:
            counts[value] += 1
        else:
            counts[value] = 1

    # Déterminer le minimum et le maximum pour compléter les nombres manquants
    min_val = min(course)
    max_val = max(course)

    # Ajouter les nombres manquants avec un comptage de 0
    for i in range(min_val, max_val + 1):
        if i not in counts:
            counts[i] = 0

    # Retourner un dictionnaire trié par clé (nombre)
    return dict(sorted(counts.items()))

#Test du carré-unité (Prendre nombre 4 a 4 pour en faire un graphique)
def carreUnit(a,c,m,x0, size):
    yn = ynCompute(a,c,m,x0)
    carreUnit = []
    for i in range(0, size*4, 4):
        carreUnit.append((yn[i+2] - yn[i])**2 + (yn[i+3] - yn[i+1])**2)
    return carreUnit

def poisson(lmbda):
    eps=1e-6
    probs = []
    unCumulated = []
    total = 0.0
    k = 0

    while (1 - total > eps) and (k <= 100000):
        p = math.exp(-lmbda) * (lmbda ** k) / math.factorial(k)
        total += p
        probs.append(p)
        unCumulated.append(total)
        k += 1

    return unCumulated

def poissonWithRng(poisson, suite):
    number = []
    for i in range(len(suite)):
        j = 0
        while(j< len(poisson) and poisson[j] < suite[i]):
            j+=1
        number.append(j)
    return number

#K = inverse d'un de modulo "m" (Permet de trouvé le k pour avoir le x0 a partir de x1)
def kCompute(x1, modulo, a, c):
    k=0
    while((x1 - c + k * modulo) % a != 0):
        k+=1
    return k

def grouping(xi,ri,pi,npi,x):
    i = 0
    while i < len(npi) - 1:
        if npi[i] < 5:
            xi[i] += " + " + xi[i+1]
            ri[i] += ri[i+1]
            pi[i] += pi[i+1]
            npi[i] += npi[i+1]
            x[i] += x[i+1]
            del xi[i+1], ri[i+1], pi[i+1], npi[i+1], x[i+1]
            if i > 0:
                i-=1
        else:
            i+=1
    if len(npi) == 1 and npi[0] <5:
        return False
    return xi, ri, pi, npi, x


In [23]:
def frequence(a,c,m,x0):
    print("Etape 1 :")
    print("H0 : chaque chiffre apparait avec la meme frequence")
    print("H1 : la distribution diffère de l'uniforme")

    print("\nEtape 2 :")
    alpha = 0.05
    print(alpha)

    print("\nEtape 3 :")
    yn = ynCompute(a, c, m, x0)  

    xi = ["0","1","2","3","4","5","6","7","8","9"]
    ri = list(counting(yn).values())
    pi = [1/len(xi) for i in xi]
    npi = [sum(ri)/len(xi) for i in xi]
    i = [(ri[j] - npi[j])**2 / npi[j] for j in range(len(ri))]
    
    df_tab = pd.DataFrame({'xi': xi, 'ri': ri, 'pi': pi, 'npi': npi, '(ri-npi)²/npi': i})

    total = pd.DataFrame({'xi': ['Total'], 'ri': [df_tab['ri'].sum()], 'pi': [df_tab['pi'].sum()], 'npi': [df_tab['npi'].sum()], '(ri-npi)²/npi': [df_tab['(ri-npi)²/npi'].sum()]})
    
    df_tab_total = pd.concat([df_tab, total])
    print(df_tab_total.to_string(index=False))

    print("\nEtape 4 :")
    if df_tab['npi'][0] >= 5 and sum(ri) > 50:
        print("La condition est respectée pas besoin de regrouper")
        nb_modalite = df_tab['ri'].sum()
    else :
        grouping(xi, ri, pi, npi, i)
        dfg_tab = pd.DataFrame({'xi': xi, 'ri': ri, 'pi' : pi, "npi" : npi, '(ri-npi)²/npi': i})
        print(dfg_tab.to_string(index=False))
        nb_modalite = dfg_tab['ri'].sum()
    print("\nEtape 5 :")
    degre_liberte = nb_modalite - 1
    x2_obs_total = df_tab['(ri-npi)²/npi'].sum()

    valeur_critique = chi2.ppf(1 - alpha, degre_liberte)

    print(f"{x2_obs_total:.2f} =< {valeur_critique:.2f}")

    print("\nEtape 6 :")
    if x2_obs_total <= valeur_critique:
        print("H0 est acceptée : chaque chiffre apparait avec la même fréquence")
    else:
        print("H1 est acceptée : la distribution diffère de l'uniforme")

def poker(a,c,m,x0):
    print("Etape 1 :")
    print("H0 : la distribution des combinaisons (paire, double paire, brelan, etc.) correspond aux probabilités théoriques.")
    print("H1 : la distribution diffère de celle attendue")

    print("\nEtape2 :")
    alpha = 0.05
    print(alpha)

    print("\nEtape 3 :")
    
    xi = ["Poker", "Carré", "Full", "Brelan", "Deux Pair", "Une Pair", "Rien"]
    ri = pokerCount(a,c,m,x0)
    pi = [1/(10**4), 450/(10**5), 900/(10**5), 7200/(10**5), 10800/(10**5), 50400/(10**5), 0.3024]
    npi = [sum(ri)*p for p in pi]
    i = [(ri[j] - npi[j])**2 / npi[j] for j in range(len(ri))]

    dp_tab = pd.DataFrame({'xi': xi, 'ri': ri, 'pi' : pi, "npi" : npi, '(ri-npi)²/npi': i})
    print(dp_tab.to_string(index=False))

    print("\nEtape 4 :")
    grouped = grouping(xi,ri,pi,npi,i)
    if(grouped):
        dpg_tab = pd.DataFrame({'xi': xi, 'ri': ri, 'pi' : pi, "npi" : npi, '(ri-npi)²/npi': i})
        print(dpg_tab.to_string(index=False))
        nb_modalite = dpg_tab['ri'].sum()
        degre_liberte = nb_modalite - 1
        x2_obs_total = dpg_tab['(ri-npi)²/npi'].sum()
        print("\nX2 = %f" % x2_obs_total)
    else:
        print("Toujours inférieur a 5 aprés regroupement de toutes les catégories")
        nb_modalite = dp_tab['ri'].sum()
        degre_liberte = nb_modalite - 1
        x2_obs_total = dp_tab['(ri-npi)²/npi'].sum()
        print("\nX2 = %f" % x2_obs_total)
    
    print("\nEtape 5 :")

    valeur_critique = chi2.ppf(1 - alpha, degre_liberte)
    print("\nv = %f" % valeur_critique)

    print("\nEtape 6 :")
    
    if x2_obs_total <= valeur_critique:
        print("H0 est acceptée : la distribution des combinaisons (paire, double paire, brelan, etc.) correspond aux probabilités théoriques.")
    else:
        print("H1 est acceptée : la distribution diffère de celle attendue")
        
def partie1(a,c,m,x0):
    full_period = isFullPeriod(a, c, m)

    if not full_period:
        print("Le 3 hypothèses du théorème de Hull-Dobell ne sont pas respectées")
    else:
        print("Les 3 hypothèses du théorème de Hull-Dobell sont respectées")
        print("\nTest de fréquence en six étapes :\n")
        frequence(a,c,m,x0)
        print("\nTest de poker en six étapes :\n")
        poker(a,c,m,x0)
partie1(1597, 51749, 244944, 1)

Les 3 hypothèses du théorème de Hull-Dobell sont respectées

Test de fréquence en six étapes :

Etape 1 :
H0 : chaque chiffre apparait avec la meme frequence
H1 : la distribution diffère de l'uniforme

Etape 2 :
0.05

Etape 3 :
   xi     ri  pi      npi  (ri-npi)²/npi
    0  24495 0.1  24494.4       0.000015
    1  24494 0.1  24494.4       0.000007
    2  24495 0.1  24494.4       0.000015
    3  24494 0.1  24494.4       0.000007
    4  24494 0.1  24494.4       0.000007
    5  24495 0.1  24494.4       0.000015
    6  24494 0.1  24494.4       0.000007
    7  24495 0.1  24494.4       0.000015
    8  24494 0.1  24494.4       0.000007
    9  24494 0.1  24494.4       0.000007
Total 244944 1.0 244944.0       0.000098

Etape 4 :
La condition est respectée pas besoin de regrouper

Etape 5 :
0.00 =< 246095.40

Etape 6 :
H0 est acceptée : chaque chiffre apparait avec la même fréquence

Test de poker en six étapes :

Etape 1 :
H0 : la distribution des combinaisons (paire, double paire, brelan, etc

In [24]:
import math
import pandas as pd


def partie2(a, c, m, x0):
    """Partie 2 du projet - Simulation du système d'attente"""
    
    # Générer la séquence pseudo-aléatoire complète
    sequence_aleatoire = xnCompute(a, c, m, x0)
    indice_sequence = 0
    
    def obtenir_nombre_aleatoire():
        """Retourne le prochain nombre de la séquence pseudo-aléatoire normalisé entre 0 et 1"""
        nonlocal indice_sequence
        valeur = sequence_aleatoire[indice_sequence % len(sequence_aleatoire)]
        indice_sequence += 1
        return valeur / m
    
    # Configuration
    loi_duree_service = pd.DataFrame({
        "Durée en minutes": [1, 2, 3, 4, 5, 6],
        "Répétition": [24, 18, 10, 3, 3, 2]
    })
    
    COUTS = {
        "presence_ord": 15 / 60,
        "presence_pr_rel": 35 / 60,
        "presence_pr_abs": 45 / 60,
        "occup_pr": 33 / 60,
        "occup_ord": 28 / 60,
        "inoccup": 18 / 60,
        "perte_pr": 20,
        "perte_ord": 15
    }
    
    def calcul_min_station():
        client = 1.5
        client_prio = 0.7
        total_activite = client + client_prio
        somme_répétition = loi_duree_service["Répétition"].sum()
        durée_moyenne_service = sum(loi_duree_service["Durée en minutes"] * loi_duree_service["Répétition"]) / somme_répétition
        calcul_psy = total_activite * durée_moyenne_service
        rounded_value = round(calcul_psy)
        return rounded_value
    
    
    class Client:
        def __init__(self, id_client, type_client, temps_arrivee, duree_service):
            self.id = id_client
            self.type = type_client
            self.temps_arrivee = temps_arrivee
            self.duree_service = duree_service
            self.duree_restante = duree_service
            self.temps_debut_service = None
        
        def __repr__(self):
            return f"C{self.id}({self.type[0].upper()}, durée={self.duree_service})"
    
    class Station:
        def __init__(self, id_station):
            self.id = id_station
            self.client = None
        
        def est_libre(self):
            return self.client is None
        
        def affecter_client(self, client, temps_actuel):
            self.client = client
            client.temps_debut_service = temps_actuel
        
        def traiter_minute(self):
            if self.client:
                self.client.duree_restante -= 1
                if self.client.duree_restante <= 0:
                    client_libere = self.client
                    self.client = None
                    return client_libere
            return None
        
        def __repr__(self):
            if self.client:
                return f"S{self.id}[{self.client}, reste={self.client.duree_restante}min]"
            return f"S{self.id}[Libre]"
    
    class SystemeAttente:
        def __init__(self, nb_stations, lambda_ordinaire=1.5, lambda_prioritaire=0.7, 
                     proba_absolu=0.3, duree_simulation=200):
            self.nb_stations = nb_stations
            self.lambda_ordinaire = lambda_ordinaire
            self.lambda_prioritaire = lambda_prioritaire
            self.proba_absolu = proba_absolu
            self.duree_simulation = duree_simulation
            
            # Loi de service
            self.durees_service = [1, 2, 3, 4, 5, 6]
            self.repetitions = [24, 18, 10, 3, 3, 2]
            self.proba_service = [r/sum(self.repetitions) for r in self.repetitions]
            
            # Coûts
            self.cout_presence_ordinaire = 15/60
            self.cout_presence_relatif = 35/60
            self.cout_presence_absolu = 45/60
            self.cout_occupation_ordinaire = 28/60
            self.cout_occupation_prioritaire = 33/60
            self.cout_inoccupation = 18/60
            
            # État du système
            self.stations = [Station(i) for i in range(nb_stations)]
            self.file_absolus = []
            self.file_relatifs = []
            self.file_ordinaires = []
            
            # Statistiques
            self.compteur_clients = 0
            self.cout_total_presence = 0
            self.cout_total_occupation = 0
            self.cout_total_inoccupation = 0
            self.clients_termines = []
        
        def generer_arrivees_poisson(self, lambda_param):
            """Génère un temps inter-arrivée selon une loi de Poisson"""
            u = obtenir_nombre_aleatoire()
            return -math.log(1 - u) / lambda_param if u < 1 else 0
        
        def generer_duree_service(self):
            """Génère une durée de service selon la distribution donnée"""
            u = obtenir_nombre_aleatoire()
            cumul = 0
            for i, prob in enumerate(self.proba_service):
                cumul += prob
                if u <= cumul:
                    return self.durees_service[i]
            return self.durees_service[-1]
        
        def generer_clients_minute(self, minute):
            """Génère les clients arrivant à une minute donnée"""
            clients = []
            
            # Clients ordinaires
            temps_inter_arrivee = self.generer_arrivees_poisson(self.lambda_ordinaire)
            if temps_inter_arrivee < 1:
                self.compteur_clients += 1
                duree = self.generer_duree_service()
                clients.append(Client(self.compteur_clients, 'ordinaire', minute, duree))
            
            # Clients prioritaires
            temps_inter_arrivee = self.generer_arrivees_poisson(self.lambda_prioritaire)
            if temps_inter_arrivee < 1:
                self.compteur_clients += 1
                duree = self.generer_duree_service()
                type_client = 'prioritaire_absolu' if obtenir_nombre_aleatoire() < self.proba_absolu else 'prioritaire_relatif'
                clients.append(Client(self.compteur_clients, type_client, minute, duree))
            
            return clients
        
        def afficher_etat(self, minute, phase):      
            print(f"Minute {minute} - {phase}")
            print("\nStations:")
            for station in self.stations:
                print(f"  {station}")
            
            print(f"\nFiles d'attente:")
            print(f"  Prioritaires absolus ({len(self.file_absolus)}): {self.file_absolus}")
            print(f"  Prioritaires relatifs ({len(self.file_relatifs)}): {self.file_relatifs}")
            print(f"  Ordinaires ({len(self.file_ordinaires)}): {self.file_ordinaires}")
        
        def placer_clients(self, minute):
            """Place les clients en attente dans les stations libres"""
            files_ordonnees = [
                (self.file_absolus, 'prioritaire_absolu'),
                (self.file_relatifs, 'prioritaire_relatif'),
                (self.file_ordinaires, 'ordinaire')
            ]
            
            for file, type_client in files_ordonnees:
                while file:
                    station_libre = next((s for s in self.stations if s.est_libre()), None)
                    if station_libre:
                        client = file.pop(0)
                        station_libre.affecter_client(client, minute)
                    else:
                        break
        
        def calculer_couts_minute(self):
            """Calcule les coûts pour la minute en cours"""
            # Coût de présence dans les files
            for client in self.file_absolus:
                self.cout_total_presence += self.cout_presence_absolu
            for client in self.file_relatifs:
                self.cout_total_presence += self.cout_presence_relatif
            for client in self.file_ordinaires:
                self.cout_total_presence += self.cout_presence_ordinaire
            
            # Coût d'occupation et inoccupation des stations
            for station in self.stations:
                if station.client:
                    self.cout_total_presence += (
                        self.cout_presence_absolu if station.client.type == 'prioritaire_absolu'
                        else self.cout_presence_relatif if station.client.type == 'prioritaire_relatif'
                        else self.cout_presence_ordinaire
                    )
                    self.cout_total_occupation += (
                        self.cout_occupation_prioritaire if 'prioritaire' in station.client.type
                        else self.cout_occupation_ordinaire
                    )
                else:
                    self.cout_total_inoccupation += self.cout_inoccupation
        
        def simuler(self, afficher_details=False):
            """Lance la simulation du système d'attente"""
            print(f"\nSIMULATION AVEC {self.nb_stations} STATIONS\n")
            
            for minute in range(1, self.duree_simulation + 1):
                if afficher_details and minute <= 20:
                    self.afficher_etat(minute, "DÉBUT DE MINUTE")
                
                # Traiter les stations
                for station in self.stations:
                    client_termine = station.traiter_minute()
                    if client_termine:
                        self.clients_termines.append(client_termine)
                
                # Générer les nouvelles arrivées
                nouveaux_clients = self.generer_clients_minute(minute)
                
                if afficher_details and minute <= 20:
                    print(f"\nARRIVÉES à la minute {minute}:\n")
                    if nouveaux_clients:
                        for client in nouveaux_clients:
                            print(f"  {client}")
                    else:
                        print("  Aucune arrivée")
                    self.afficher_etat(minute, "AVANT PLACEMENT")
                
                # Ajouter aux files
                for client in nouveaux_clients:
                    if client.type == 'prioritaire_absolu':
                        self.file_absolus.append(client)
                    elif client.type == 'prioritaire_relatif':
                        self.file_relatifs.append(client)
                    else:
                        self.file_ordinaires.append(client)
                
                # Placer les clients
                self.placer_clients(minute)
                
                if afficher_details and minute <= 20:
                    self.afficher_etat(minute, "APRÈS PLACEMENT")
                
                # Calculer les coûts
                self.calculer_couts_minute()
                
                if afficher_details and minute <= 20:
                    self.afficher_etat(minute, "FIN DE MINUTE")
            
            self.afficher_resultats()
            return self.cout_total_presence + self.cout_total_occupation + self.cout_total_inoccupation
        
        def afficher_resultats(self):
            """Affiche les résultats finaux de la simulation"""
            cout_total = self.cout_total_presence + self.cout_total_occupation + self.cout_total_inoccupation
            
            print(f"RÉSULTATS FINAUX - {self.nb_stations} STATIONS")
            print(f"Coût total de présence dans le système: {self.cout_total_presence:.2f} €")
            print(f"Coût total d'occupation des stations: {self.cout_total_occupation:.2f} €")
            print(f"Coût total d'inoccupation des stations: {self.cout_total_inoccupation:.2f} €")
            print(f"---------------------------------")
            print(f"COÛT TOTAL: {cout_total:.2f} €")
            print(f"---------------------------------")
            print(f"Clients traités: {len(self.clients_termines)}")
            print(f"Clients en attente: {len(self.file_absolus) + len(self.file_relatifs) + len(self.file_ordinaires)}")
            print(f"Clients en service: {sum(1 for s in self.stations if not s.est_libre())}")
            print(f"\n")
    
    # Programme principal de la partie 2
    
    # Calculer le minimum de stations
    min_stations = calcul_min_station()
    print(f"\nNombre minimum de stations calculé: {min_stations}")
    
    # Tester différentes configurations
    nb_stations_a_tester = [5, 6, 7, 8, 9, 10]
    resultats = {}
    
    for nb_stations in nb_stations_a_tester:
        # Réinitialiser l'indice de la séquence pour chaque simulation
        indice_sequence = 0
        
        systeme = SystemeAttente(
            nb_stations=nb_stations,
            lambda_ordinaire=1.5,
            lambda_prioritaire=0.7,
            proba_absolu=0.3,
            duree_simulation=200
        )
        
        afficher = (nb_stations == 5)
        cout_total = systeme.simuler(afficher_details=afficher)
        resultats[nb_stations] = cout_total
    
    # Afficher le récapitulatif
    print("RÉCAPITULATIF")
    print(f"------------------------------")
    print(f"{'Nb Stations':<15} {'Coût Total (€)':<20}")
    print(f"------------------------------")
    for nb, cout in sorted(resultats.items()):
        print(f"{nb:<15} {cout:<20.2f}")
    
    nb_optimal = min(resultats, key=resultats.get)
    print(f"------------------------------")
    print(f"Configuration optimale: {nb_optimal} stations")
    print(f"Coût minimal: {resultats[nb_optimal]:.2f} €")
    print(f"------------------------------")


# Exécution
partie2(1597, 51749, 244944, 1)


Nombre minimum de stations calculé: 5

SIMULATION AVEC 5 STATIONS

Minute 1 - DÉBUT DE MINUTE

Stations:
  S0[Libre]
  S1[Libre]
  S2[Libre]
  S3[Libre]
  S4[Libre]

Files d'attente:
  Prioritaires absolus (0): []
  Prioritaires relatifs (0): []
  Ordinaires (0): []

ARRIVÉES à la minute 1:

  C1(O, durée=1)
  C2(P, durée=2)
Minute 1 - AVANT PLACEMENT

Stations:
  S0[Libre]
  S1[Libre]
  S2[Libre]
  S3[Libre]
  S4[Libre]

Files d'attente:
  Prioritaires absolus (0): []
  Prioritaires relatifs (0): []
  Ordinaires (0): []
Minute 1 - APRÈS PLACEMENT

Stations:
  S0[C2(P, durée=2), reste=2min]
  S1[C1(O, durée=1), reste=1min]
  S2[Libre]
  S3[Libre]
  S4[Libre]

Files d'attente:
  Prioritaires absolus (0): []
  Prioritaires relatifs (0): []
  Ordinaires (0): []
Minute 1 - FIN DE MINUTE

Stations:
  S0[C2(P, durée=2), reste=2min]
  S1[C1(O, durée=1), reste=1min]
  S2[Libre]
  S3[Libre]
  S4[Libre]

Files d'attente:
  Prioritaires absolus (0): []
  Prioritaires relatifs (0): []
  Ordinaire